In [179]:
from pathlib import Path
import sys
import os
import importlib


# Loading markov model directory as module into jupyter notebook
module_dir = Path(os.getcwd()).parent.resolve()
data_dir = module_dir / "data"

if module_dir not in sys.path:
    sys.path.insert(1, str(module_dir))
else:
    print("Module path already inserted into system paths")

try:
    from model import markov
    from model import constants
    from model import utils

    # to apply changes in modules
    importlib.reload(markov)
    importlib.reload(constants)
    importlib.reload(utils)
except ModuleNotFoundError as e:
    print(f"Unable to import module: {e.msg}")

In [180]:
# Model inputs
n_samples = 2
short_steps = constants.SHORT_TERM_CYCLE_COUNTS
long_steps = constants.LONG_TERM_CYCLE_COUNTS
# Loading states name from excel sheet is deprecated, now on only generate within the code blocks
start_state = "Healthy"
primary_states = [
    "Healthy",
    "Bleeding",
    "Hemarthrosis",
    "Arthropathy",
    "LT_Bleeding",
    "Death",
]
secondary_states = ["Healthy", "Bleeding", "Hemarthrosis", "LT_Bleeding", "Death"]
# NOTE:
# Transition matrix is dynamically generated through the *_psa functions
# To change in states need to update the psa worker functions as well to support new model schema
# NOTE:
# Newly suggested model structure:               switch
#                   [Healthy]                    ------>                     [Arthropathy]
#        |              |              |                           |              |              |
# [LT Bleeding] | [Hemarthrosis] | [Bleeding]               [LT Bleeding] | [Hemarthrosis] | [Bleeding]
#    |                  |                                          |
# [DEATH]         [Arthropathy]                                 [DEATH]

chains = {"primary": (primary_states, {}), "secondary": (secondary_states, {})}


# Define switch conditions
def arthropathy_switch_condition(step: int, state: str, chain: str, **kwargs) -> bool:
    """Determine if a switch to the secondary chain should occur based on the Arthropathy state."""
    return state == "Arthropathy" and chain == "primary"


switch_conditions = {"secondary": arthropathy_switch_condition}

In [181]:
weights = [utils.cal_body_weight(w, b=2 * 52) for w in range(short_steps)]
# Short term simulation
on_demand_inputs, on_demand_outputs = markov.psa_simulation(
    strategy="on_demand",
    n_samples=n_samples,
    chains=chains,
    start_chain="primary",
    start_state="Healthy",
    steps=short_steps,
    switch_conditions=switch_conditions,
)
prophylaxis_inputs, prophylaxis_outputs = markov.psa_simulation(
    strategy="prophylaxis",
    n_samples=n_samples,
    chains=chains,
    start_chain="primary",
    start_state="Healthy",
    steps=short_steps,
    switch_conditions=switch_conditions,
)

In [182]:
import numpy as np
import pandas as pd


df = pd.DataFrame(
    columns=[
        "mean_on_demand",
        "median_on_demand",
        "mean_prophylaxis",
        "median_prophylaxis",
    ],
)

# NOTE:
# QALYs values are far less than expectation, which seems there is a mistake some where on markov chain switching
# Transition matrix generator or utility reward function implementation.
# Debug before continuing

print(f"[ Short Term ] Simulation results for {int(short_steps/52)} Years (2 - 13)")

# ON_DEMAND
df.loc["Annual Factor Consumption", "mean_on_demand"] = (
    np.mean(on_demand_outputs["total_factors_use"]) / 10
)
df.loc["Annual Factor Consumption", "median_on_demand"] = (
    np.median(on_demand_outputs["total_factors_use"]) / 10
)
# PER KG
df.loc["Annual Factor Consumption per Kg", "mean_on_demand"] = (
    np.mean(on_demand_outputs["total_factors_use"]) / 10
) / np.mean(weights)
df.loc["Annual Factor Consumption per Kg", "median_on_demand"] = (
    np.median(on_demand_outputs["total_factors_use"]) / 10
) / np.median(weights)

# QALY
df.loc["QALY Gained", "mean_on_demand"] = np.mean(on_demand_outputs["QALYS"])
df.loc["QALY Gained", "median_on_demand"] = np.median(on_demand_outputs["QALYS"])

# PROPHYLAXIS
df.loc["Annual Factor Consumption", "mean_prophylaxis"] = (
    np.mean(prophylaxis_outputs["total_factors_use"]) / 10
)
df.loc["Annual Factor Consumption", "median_prophylaxis"] = (
    np.median(prophylaxis_outputs["total_factors_use"]) / 10
)
# PER KG
df.loc["Annual Factor Consumption per Kg", "mean_prophylaxis"] = (
    np.mean(prophylaxis_outputs["total_factors_use"]) / 10
) / np.mean(weights)
df.loc["Annual Factor Consumption per Kg", "median_prophylaxis"] = (
    np.median(prophylaxis_outputs["total_factors_use"]) / 10
) / np.median(weights)

# QALY
df.loc["QALY Gained", "mean_prophylaxis"] = np.mean(prophylaxis_outputs["QALYS"])
df.loc["QALY Gained", "median_prophylaxis"] = np.median(prophylaxis_outputs["QALYS"])
df

[ Short Term ] Simulation results for 10 Years (2 - 13)


,mean_on_demand,median_on_demand,mean_prophylaxis,median_prophylaxis
Annual Factor Consumption,37137.716667,41251.3,134513.0,129911.8
Annual Factor Consumption per Kg,1530.10899,1733.976461,5542.062598,5460.773434
QALY Gained,4.880769,6.547115,5.626522,6.569423
